# import 

In [41]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

# 데이터 로드

In [42]:
# 0) 데이터 로드
df = pd.read_csv("diabetes_prediction_dataset.csv", encoding="cp949")
df.info() # 데이터 타입 확인 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   성별        100000 non-null  object 
 1   나이        100000 non-null  float64
 2   고혈압 여부    100000 non-null  int64  
 3   심장질환 여부   100000 non-null  int64  
 4   흡연 경험     100000 non-null  object 
 5   BMI 지수    100000 non-null  float64
 6   당화혈색소 수치  100000 non-null  float64
 7   혈당 수치     100000 non-null  int64  
 8   당뇨병 여부    100000 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 6.9+ MB


# 데이터 분할 

In [43]:
# 1) 데이터 분할 (8:2)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["당뇨병 여부"]
)

target_col = "당뇨병 여부"
X_train_raw = train_df.drop(target_col, axis=1).copy() # 학습 세트의 독립변수 지정
y_train = train_df[target_col].copy() # 학습 세트의 종속변수 지정
X_test_raw = test_df.drop(target_col, axis=1).copy() # 테스트 세트의 독립변수 지정
y_test = test_df[target_col].copy() # 테스트 세트의 종속변수 지정

# 1. 데이터 크기(행, 열) 확인: 데이터가 의도대로 잘 쪼개졌는지 체크
print(f"학습용 데이터 크기: {X_train_raw.shape}, 정답: {y_train.shape}")
print(f"테스트용 데이터 크기: {X_test_raw.shape}, 정답: {y_test.shape}")

# 2. 타깃(종속변수) 비율 확인: stratify가 잘 작동해서 비율이 깨지지 않았는지 체크
print(f"\n학습 데이터 타깃 비율:\n{y_train.value_counts(normalize=True)}")


학습용 데이터 크기: (80000, 8), 정답: (80000,)
테스트용 데이터 크기: (20000, 8), 정답: (20000,)

학습 데이터 타깃 비율:
당뇨병 여부
0    0.915
1    0.085
Name: proportion, dtype: float64


# 데이터 전처리 

1. 결측치 처리 -> 결측치가 없어서 안해도 됨

In [44]:
# 컬럼별 결측치 개수 확인
print(df.isnull().sum())


성별          0
나이          0
고혈압 여부      0
심장질환 여부     0
흡연 경험       0
BMI 지수      0
당화혈색소 수치    0
혈당 수치       0
당뇨병 여부      0
dtype: int64


2. 이상치 처리 -> [당화혈색소 수치], [혈당 수치]는 "이상치 내부 당뇨 환자 분포"가 100% 이므로 이상치가 아닌 유의미한 데이터. 

따라서 [BMI 지수]에 대해서만 이상치 처리 진행. 

In [45]:
# 분석할 컬럼 목록
target_cols = ['BMI 지수', '당화혈색소 수치', '혈당 수치']

def analyze_outliers_detailed(df, col):
    """
    특정 컬럼에 대해 IQR 기반 이상치를 분석하고 리포트를 출력하는 함수
    """
    # 1) IQR 및 정상 범위 계산
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    print(f"■ [{col}]")
    print(f"1) 정상 범위: {lower_bound:.2f} ~ {upper_bound:.2f}")
    
    # 2) 이상치 데이터 발색 (필터링)
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    total_count = len(df)
    outlier_count = len(outliers)
    
    print(f"2) 이상치 개수: {outlier_count}개")
    
    # 3) 이상치 중 당뇨(Positive) 비율 분석
    if outlier_count > 0:
        diabetes_in_outliers = outliers[outliers['당뇨병 여부'] == 1]
        diabetes_count = len(diabetes_in_outliers)
        diabetes_ratio = (diabetes_count / outlier_count) * 100
        
        print(f"3) 이상치 내부 당뇨 환자 분포:")
        print(f"   - 당뇨 환자 수: {diabetes_count}명")
        print(f"   - 당뇨 비율: {diabetes_ratio:.2f}%")
        
        # 비교를 위해 전체 데이터의 당뇨 비율도 함께 출력하면 좋습니다
        total_diabetes_ratio = (df['당뇨병 여부'].sum() / total_count) * 100
    else:
        print("3) 발견된 이상치가 없습니다.")
        
    print("=" * 50)  # 구분선

# 실행
for col in target_cols:
    analyze_outliers_detailed(df, col)

■ [BMI 지수]
1) 정상 범위: 14.71 ~ 38.50
2) 이상치 개수: 7086개
3) 이상치 내부 당뇨 환자 분포:
   - 당뇨 환자 수: 1478명
   - 당뇨 비율: 20.86%
■ [당화혈색소 수치]
1) 정상 범위: 2.70 ~ 8.30
2) 이상치 개수: 1315개
3) 이상치 내부 당뇨 환자 분포:
   - 당뇨 환자 수: 1315명
   - 당뇨 비율: 100.00%
■ [혈당 수치]
1) 정상 범위: 11.50 ~ 247.50
2) 이상치 개수: 2038개
3) 이상치 내부 당뇨 환자 분포:
   - 당뇨 환자 수: 2038명
   - 당뇨 비율: 100.00%


In [46]:
# 1. 학습 데이터(X_train_raw) 기준으로 BMI 정상 범위 상한값(Upper Limit) 계산
# 주의: 테스트 데이터(test_df)는 절대 계산에 포함시키면 안 됨 (Data Leakage 방지)
bmi_col = "BMI 지수"

Q1 = X_train_raw[bmi_col].quantile(0.25)
Q3 = X_train_raw[bmi_col].quantile(0.75)
IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR

print(f"✅ BMI 상한값(Upper Limit): {upper_limit:.2f}")

# 2. 학습 데이터(X_train_raw)에 적용: 상한값보다 큰 값은 상한값으로 대체 (Clipping)
# 예: 상한이 40인데 50인 값이 있으면 40으로 변경
X_train_raw[bmi_col] = X_train_raw[bmi_col].clip(upper=upper_limit)

# 3. 테스트 데이터(X_test_raw)에도 '학습 데이터에서 구한 기준' 그대로 적용
# 테스트 데이터만의 IQR을 새로 구하면 안 됨!
X_test_raw[bmi_col] = X_test_raw[bmi_col].clip(upper=upper_limit)

print(">> 이상치 처리 완료: 상한값 초과 데이터를 상한값으로 대체했습니다.")

# (선택) 처리 후 최댓값이 상한값과 같아졌는지 확인
print(f"처리 후 Train BMI 최대값: {X_train_raw[bmi_col].max()}")
print(f"처리 후 Test BMI 최대값: {X_test_raw[bmi_col].max()}")


✅ BMI 상한값(Upper Limit): 38.43
>> 이상치 처리 완료: 상한값 초과 데이터를 상한값으로 대체했습니다.
처리 후 Train BMI 최대값: 38.43000000000001
처리 후 Test BMI 최대값: 38.43000000000001


3. 범주형 변수 인코딩

In [47]:
cat_cols = ["성별", "흡연 경험"]

for col in cat_cols:
    print(f"■ [{col}] 카테고리 분포")
    # dropna=False를 넣으면 결측치(NaN)가 있는지도 함께 보여줍니다
    print(X_train_raw[col].value_counts(dropna=False))
    print("-" * 30)


■ [성별] 카테고리 분포
성별
Female    46810
Male      33174
Other        16
Name: count, dtype: int64
------------------------------
■ [흡연 경험] 카테고리 분포
흡연 경험
No Info        28579
never          28082
former          7530
current         7443
not current     5182
ever            3184
Name: count, dtype: int64
------------------------------


In [48]:
# 1. 'Other' 데이터 삭제 (전체 데이터 대비 극소량이므로 삭제가 유리)
# 원본 데이터 df에서 처리하거나, train_df/test_df 분할 전에 처리하는 게 가장 좋습니다.
# 여기서는 X_train_raw 단계라고 가정하고 필터링합니다.

print(f"삭제 전 크기: {X_train_raw.shape}")

# 성별이 'Other'가 아닌 것만 남기기
X_train_raw = X_train_raw[X_train_raw['성별'] != 'Other'].copy()
X_test_raw = X_test_raw[X_test_raw['성별'] != 'Other'].copy()

# 삭제 후 y(정답지) 데이터도 인덱스 맞춰서 줄여줘야 함 (매우 중요!)
y_train = y_train[X_train_raw.index]
y_test = y_test[X_test_raw.index]

print(f"삭제 후 크기: {X_train_raw.shape}")

# 2. 성별을 0과 1로 매핑 (이제 카테고리가 2개뿐이므로 순서 문제 없음)
gender_map = {'Female': 0, 'Male': 1}

X_train_raw['성별'] = X_train_raw['성별'].map(gender_map)
X_test_raw['성별'] = X_test_raw['성별'].map(gender_map)

# 3. 데이터 타입 확인 (정수형으로 잘 바뀌었는지)
print("\n[변환 후 성별 분포]")
print(X_train_raw['성별'].value_counts())


삭제 전 크기: (80000, 8)
삭제 후 크기: (79984, 8)

[변환 후 성별 분포]
성별
0    46810
1    33174
Name: count, dtype: int64


In [49]:
# 타깃 범주형 컬럼 지정
target_cat_col = ["흡연 경험"]

# 1. 인코더 생성
# handle_unknown='ignore': 학습 때 없던 새로운 흡연 상태가 테스트 때 나오면 에러 없이 0으로 처리
# sparse_output=False: 결과를 바로 보기 편한 배열(Array) 형태로 받음
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# 2. 학습 데이터(X_train_raw)로 학습 및 변환
# 주의: 대괄호 두 개 [['흡연 경험']]를 써서 2차원 DataFrame 형태로 넣어야 함
train_smoking_encoded = encoder.fit_transform(X_train_raw[target_cat_col])

# 3. 테스트 데이터(X_test_raw) 변환 (Transform Only)
test_smoking_encoded = encoder.transform(X_test_raw[target_cat_col])

# 4. 인코딩된 결과를 DataFrame으로 예쁘게 만들기
# get_feature_names_out(): ['흡연 경험_never', '흡연 경험_current', ...] 처럼 이름 자동 생성
new_cols = encoder.get_feature_names_out(target_cat_col)

train_smoking_df = pd.DataFrame(train_smoking_encoded, columns=new_cols, index=X_train_raw.index)
test_smoking_df = pd.DataFrame(test_smoking_encoded, columns=new_cols, index=X_test_raw.index)

# 5. 기존 데이터와 합치기 (Concat)
# 기존 X_train_raw에서 원본 '흡연 경험' 컬럼은 이제 필요 없으니 제거(drop)하고
# 새로 만든 원-핫 인코딩 데이터프레임(train_smoking_df)을 옆으로 붙임
X_train_final = pd.concat([X_train_raw.drop(columns=target_cat_col), train_smoking_df], axis=1)
X_test_final = pd.concat([X_test_raw.drop(columns=target_cat_col), test_smoking_df], axis=1)

print("✅ '흡연 경험' 원-핫 인코딩 완료!")
print(f"\n생성된 컬럼 목록: {new_cols}")

# 결과 확인
print("\n[최종 데이터 샘플]")
display(X_train_final.head())

✅ '흡연 경험' 원-핫 인코딩 완료!

생성된 컬럼 목록: ['흡연 경험_No Info' '흡연 경험_current' '흡연 경험_ever' '흡연 경험_former' '흡연 경험_never'
 '흡연 경험_not current']

[최종 데이터 샘플]


,성별,나이,고혈압 여부,심장질환 여부,BMI 지수,당화혈색소 수치,혈당 수치,흡연 경험_No Info,흡연 경험_current,흡연 경험_ever,흡연 경험_former,흡연 경험_never,흡연 경험_not current
74736,0,80.0,1,0,27.32,6.5,145,0.0,0.0,0.0,1.0,0.0,0.0
36589,0,19.0,0,0,25.18,4.5,126,0.0,0.0,0.0,0.0,1.0,0.0
37414,0,36.0,0,0,25.95,6.6,200,1.0,0.0,0.0,0.0,0.0,0.0
71251,0,35.0,0,0,23.43,6.0,159,0.0,1.0,0.0,0.0,0.0,0.0
40454,0,30.0,0,0,22.62,5.0,90,1.0,0.0,0.0,0.0,0.0,0.0
